Set 3 excercises

In [1]:
# Ex 1
N=100
# X_i~Bernoulli(p)
k=23
# i.e. X_i = 1 in 23 times out of 100 repeats

In [7]:
# Imports
import numpy as np
import jax
import pandas as pd
import jax.numpy as jnp
import jax.scipy.special as jsps
import scipy.optimize
from scipy.stats import bernoulli
from scipy.differentiate import derivative

In [3]:
# Ex 1.1 MLE estimator p hat for p
# In AoS 9.3 its stated that MLE p hat = S/n, S being the number of successes and n the number of trials. In our case, S=k and n=N, so we have:
p_hat = k/N
p_hat #0,23

0.23

In [4]:
# Ex 1.2
# 95% confidence interval for p_hat based on the lecture 6 example


In [5]:
p = 23/100 # probability of success
x = np.array([1] * 23 + [0] * 77)

log_pmf = bernoulli.logpmf(x, p)

log_likelyhood = log_pmf.sum()

# From lecture 6 slide 21-22
first_d = (x/p) - ((1-x)/(1-p))
second_d = -(x/p**2) - ((1-x)/(1-p)**2)

fisher_info = -second_d.mean()

# print(fisher_info) #5.646527385657823   

se_hat = np.sqrt(1/(N*fisher_info))

ci_lower = p_hat - 2 * se_hat
ci_upper = p_hat + 2 * se_hat

print(f"95% confidence interval for p_hat: ({ci_lower}, {ci_upper})")

#se_hat #np.float64(0.04208325082500162)

# result = scipy.optimize.minimize(
#     bernoulli_nll, x0=0.5,
#     args=(x,), method='L-BFGS-B',
#     bounds=[(0.0001, 1)])

# print(result.x)
# print(result.success)

95% confidence interval for p_hat: (0.14583349834999676, 0.31416650165000326)


In [9]:
# Ex 1.3
data = pd.Series([1] * 23 + [0] * 77)
bootstrap_estimates = []

for _ in range(10000):
    bootstrap_sample = data.sample(
        n=100,
        replace=True
    )

    bootstrap_p_hat = bootstrap_sample.mean()

    bootstrap_estimates.append(bootstrap_p_hat)

percentile_left = np.quantile(bootstrap_estimates, 0.025)
percentile_right = np.quantile(bootstrap_estimates, 0.975)
print(f"95% confidence interval for p_hat using bootstrap: ({percentile_left}, {percentile_right})")

95% confidence interval for p_hat using bootstrap: (0.15, 0.31)


In [11]:
ci_pivotal = 2 * p_hat - percentile_right, 2 * p_hat - percentile_left
print(f"95% confidence interval for p_hat using bootstrap pivotal method: {ci_pivotal}")
print(f"Pivotal CI: ({ci_pivotal[0]:.2f}, {ci_pivotal[1]:.2f})")

95% confidence interval for p_hat using bootstrap pivotal method: (np.float64(0.15000000000000002), np.float64(0.31000000000000005))
Pivotal CI: (0.15, 0.31)


In [14]:
# Ex 5
# Using lecture notes 6.3 example for MLE for a beta distribution as help.

import numpy as np
import scipy.optimize
import jax
import jax.numpy as jnp
import jax.scipy.special as jsps

x = np.loadtxt("week3_exercise5.txt", skiprows=1)

In [17]:
def invgamma_nll(theta, x):
    alpha, beta = theta

    nll = -jnp.sum(
        alpha * jnp.log(beta)
        - jsps.gammaln(alpha)
        - (alpha + 1) * jnp.log(x)
        - beta / x
    ) 

    return nll

In [21]:
invgamma_grad = jax.grad(invgamma_nll)
bounds = [(0.0001, None), (0.0001, None)]

result = scipy.optimize.minimize(
    invgamma_nll,
    x0=jnp.array([1.0, 1.0]),
    args=(x,),
    jac=invgamma_grad,
    method='L-BFGS-B',
    bounds=bounds
)

print(result)
print("alpha_hat =", result.x[0])
print("beta_hat =", result.x[1])

  message: CONVERGENCE: RELATIVE REDUCTION OF F <= FACTR*EPSMCH
  success: True
   status: 0
      fun: 2844.137
        x: [ 1.005e+00  2.015e+00]
      nit: 6
      jac: [ 3.210e-02 -6.348e-03]
     nfev: 15
     njev: 15
 hess_inv: <2x2 LbfgsInvHessProduct with dtype=float64>
alpha_hat = 1.0048654193602207
beta_hat = 2.015279816701973
